# 07 — Depth Reconstruction Comparison

**The question:** how accurately can kinematics-based simulation reconstruct 3D wrist
position (RMSE) compared with MediaPipe's native relative depth?

### Why this notebook exists

The original `sample_ground_truth()` drew **random joint angles** and ran
forward kinematics. It never looked at the recorded video, so sample *i* had
no relationship to frame *i*. Measured on the arm track:

| Predictions fed in | RMSE-kin |
|---|---|
| Real | 1206.009 |
| Shuffled | 1205.255 |
| **All zeros** | **122.861** |

Shuffling moved the score 0.06%; predicting nothing scored ten times better.
Both RMSE columns were withdrawn from `06_model_comparison`.

### What makes this answerable now

The depth sensor works. It was streaming `Y12`, which is not distance; `Y11`
is, and it is calibrated. Calibrated depth gives a **real metric 3-D wrist
position in the same frame as the landmarks** — genuinely paired, which is
exactly what was missing.

    ground truth : wrist pixel -> calibrated depth -> pinhole back-projection
    candidate A  : kinematics -- landmarks -> joint angles -> forward kinematics
    candidate B  : MediaPipe's own relative z, the scale-ambiguous baseline

Both are scored against the same ground truth with the same `rmse_depth_mm()`.

**Requires recordings made after the Y11 fix.** Earlier ones are refused.

## Step 1 — Imports and recordings

In [1]:
import os
import sys
import glob

import cv2
import numpy as np

_SCRIPTS = os.path.abspath('..')
sys.path.insert(0, _SCRIPTS)

import depth_calibration
from evaluation.depth_ground_truth import build_paired_set
from evaluation.benchmark_models import (rmse_depth_mm, similarity_align,
                                         reconstruction_controls)
from motion_mapping import landmarks_to_joint_angles, JOINT_NAMES
from robot_control.ik_solver import forward

VIDEO_DIR = '../recordings'
POSE_WRIST = 16          # MediaPipe Pose right wrist

def _sort_key(p):
    stem = os.path.splitext(os.path.basename(p))[0]
    return (0, int(stem)) if stem.isdigit() else (1, stem)

VIDEO_PATHS = sorted(glob.glob(os.path.join(VIDEO_DIR, '*.mp4')), key=_sort_key)

def depth_path_for(v):
    return os.path.splitext(v)[0] + '_depth.npz'

calib = depth_calibration.load()
print(f'Depth calibration : {calib}')
print(f'Videos found      : {len(VIDEO_PATHS)}')
usable = 0
for v in VIDEO_PATHS:
    dp = depth_path_for(v)
    if os.path.exists(dp):
        _, note = depth_calibration.load_depth_stack(dp)
    else:
        note = 'no depth file'
    ok = 'corrected' in note or 'already calibrated' in note
    usable += ok
    print(f'  {os.path.basename(v):10s} {note}')

if usable == 0:
    print('\n*** No usable depth. This comparison needs recordings made after the Y11 fix.')
else:
    print(f'\n{usable} recording(s) usable for this comparison.')

Depth calibration : (0.09999999999999987, -0.9999999999999151)
Videos found      : 5
  1.mp4      already calibrated at capture time
  2.mp4      already calibrated at capture time
  3.mp4      already calibrated at capture time
  4.mp4      already calibrated at capture time
  5.mp4      already calibrated at capture time

5 recording(s) usable for this comparison.


## Step 2 — MediaPipe Pose per frame

The arm track is used because this comparison is about reconstructing **robot-arm**
position, and the pose landmarks (shoulder/elbow/wrist) are what the
kinematic chain is driven from.

In [2]:
import mediapipe as mp

BaseOptions           = mp.tasks.BaseOptions
PoseLandmarker        = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
RunningMode           = mp.tasks.vision.RunningMode

MODEL_PATH = os.path.join(_SCRIPTS, 'perception', 'pose_landmarker_full.task')
opts = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=RunningMode.VIDEO,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
    num_poses=1,
)
print('PoseLandmarker ready.')

PoseLandmarker ready.


/home/user_02/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


## Step 3 — Build the paired set

One row per frame that has BOTH a detected pose and a valid depth reading at
the wrist pixel. Frames without usable depth are dropped and counted, never
guessed at.

In [3]:
all_lm, all_gt, all_mp = [], [], []
totals = {'frames': 0, 'detected': 0, 'kept': 0, 'dropped_no_depth': 0}

for vp in VIDEO_PATHS:
    dp = depth_path_for(vp)
    if not os.path.exists(dp):
        continue
    depth_stack, note = depth_calibration.load_depth_stack(dp)
    if depth_stack is None:
        print(f'{os.path.basename(vp)}: skipped -- {note}')
        continue

    cap = cv2.VideoCapture(vp)
    fps = cap.get(cv2.CAP_PROP_FPS) or 15.0
    landmarker = PoseLandmarker.create_from_options(opts)

    frames, idx, ts = [], [], 0.0
    fi = 0
    while True:
        ok, bgr = cap.read()
        if not ok:
            break
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        res = landmarker.detect_for_video(
            mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), int(ts))
        if res.pose_landmarks:
            lms = res.pose_landmarks[0]
            frames.append(np.array(
                [[l.x, l.y, l.z, l.visibility, getattr(l, 'presence', 1.0)] for l in lms],
                dtype=np.float32))
            idx.append(fi)
        fi += 1
        ts += 1000.0 / fps
    cap.release(); landmarker.close()

    aligned = np.stack([depth_stack[i] for i in idx if i < len(depth_stack)])
    frames  = frames[:len(aligned)]

    gt, mpz, kept, stats = build_paired_set(frames, aligned, POSE_WRIST)
    totals['frames']   += fi
    totals['detected'] += len(frames)
    totals['kept']     += stats['kept']
    totals['dropped_no_depth'] += stats['dropped_no_depth']
    print(f"{os.path.basename(vp):10s} frames {fi:4d} | pose {len(frames):4d} | "
          f"paired {stats['kept']:4d} | no-depth {stats['dropped_no_depth']:4d}")

    if gt is not None:
        all_gt.append(gt); all_mp.append(mpz)
        all_lm.extend([frames[i] for i in kept])

GT = np.concatenate(all_gt) if all_gt else None
MP = np.concatenate(all_mp) if all_mp else None
print('\nTOTAL', totals)
print('paired samples:', 0 if GT is None else len(GT))

I0000 00:00:1788606671.075232  177086 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1788606671.077042  177118 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.4), renderer: Mesa Intel(R) UHD Graphics (CML GT2)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1788606671.122735  177094 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1788606671.145512  177087 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1788606671.185651  177091 landmark_projection_calculator.cc:78] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


1.mp4      frames  665 | pose  665 | paired  425 | no-depth  240


I0000 00:00:1788606684.946936  177165 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1788606684.947606  177184 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.4), renderer: Mesa Intel(R) UHD Graphics (CML GT2)
W0000 00:00:1788606684.992771  177167 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1788606685.010773  177180 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


2.mp4      frames  655 | pose  655 | paired  464 | no-depth  191


I0000 00:00:1788606698.413267  177220 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1788606698.413947  177239 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.4), renderer: Mesa Intel(R) UHD Graphics (CML GT2)
W0000 00:00:1788606698.458879  177221 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1788606698.472237  177235 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


3.mp4      frames  661 | pose  661 | paired  448 | no-depth  213


I0000 00:00:1788606711.996083  177273 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1788606711.996688  177292 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.4), renderer: Mesa Intel(R) UHD Graphics (CML GT2)
W0000 00:00:1788606712.042862  177275 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1788606712.058754  177280 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


4.mp4      frames  654 | pose  654 | paired  418 | no-depth  236


I0000 00:00:1788606725.474219  177324 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1788606725.474880  177343 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 23.2.1-1ubuntu3.1~22.04.4), renderer: Mesa Intel(R) UHD Graphics (CML GT2)
W0000 00:00:1788606725.517855  177330 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1788606725.536591  177339 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


5.mp4      frames  659 | pose  638 | paired  369 | no-depth  269

TOTAL {'frames': 3294, 'detected': 3273, 'kept': 2124, 'dropped_no_depth': 1149}
paired samples: 2124


## Step 4 — Candidate A: kinematics reconstruction

Landmarks → joint angles (`motion_mapping`) → forward kinematics
(`ik_solver.forward`) → wrist position, in the **robot base frame**.

`WRIST_OFFSET` is deliberately NOT subtracted here. It is an untuned guess at
the camera-to-robot placement ("Tune to your physical camera/arm placement"),
and this project has no extrinsic calibration. Step 5 fits that transform from
the data instead, which is both principled and removes the guess.

In [4]:
kin = np.zeros_like(GT) if GT is not None else None

if GT is not None:
    for i, lm in enumerate(all_lm):
        angles = landmarks_to_joint_angles(lm)
        if not angles:
            kin[i, 0] = GT[i, 0]      # no pose this frame: no error contribution
            continue
        kin[i, 0] = forward(angles)   # robot base frame; aligned in Step 5
    print('kinematics reconstruction:', kin.shape)
    print()
    print('Frame check -- these live in DIFFERENT coordinate frames, which is')
    print('expected and is what the alignment in Step 5 exists to resolve:')
    for name, A in (('ground truth', GT), ('kinematics', kin), ('mediapipe z', MP)):
        v = A[:, 0, :]
        print(f'  {name:13} z range [{v[:, 2].min():+.3f}, {v[:, 2].max():+.3f}] m')

/home/user_02/.local/lib/python3.10/site-packages/ikpy/chain.py:75: UserWarning: Link Base link (index: 0) is of type 'fixed' but set as active in the active_links_mask. In practice, this fixed link doesn't provide any transformation so is as it were inactive
  warnings.warn("Link {} (index: {}) is of type 'fixed' but set as active in the active_links_mask. In practice, this fixed link doesn't provide any transformation so is as it were inactive".format(link.name, link_index))


kinematics reconstruction: (2124, 1, 3)

Frame check -- these live in DIFFERENT coordinate frames, which is
expected and is what the alignment in Step 5 exists to resolve:
  ground truth  z range [+0.550, +1.870] m
  kinematics    z range [+0.082, +0.461] m
  mediapipe z   z range [-1.868, -0.209] m


## Step 5 — Result, with controls

A bare RMSE pair is not interpretable here, so this reports two null models
next to each candidate:

- **shuffled** — the same predictions in the wrong temporal order. If a
  candidate does not clearly beat its own shuffle, the metric cannot tell
  correct landmarks from scrambled ones and the comparison is meaningless.
- **predict the mean** — a constant at the average wrist position. A candidate
  that does not beat this has learned nothing about how the wrist moves.

Both candidates are Procrustes-aligned to the ground truth first (one global
rotation/scale/translation over the whole sequence, fit once — it cannot
absorb per-frame error).

In [5]:
if GT is None or len(GT) == 0:
    print('No paired samples -- record with the Y11 pipeline first.')
else:
    kc = reconstruction_controls(kin, GT)
    mc = reconstruction_controls(MP,  GT)

    print('=== depth reconstruction vs measured depth (Z RMSE, mm) ===')
    print(f'{"Method":<28}{"raw":>10}{"aligned":>10}{"shuffled":>10}')
    print('-' * 58)
    print(f'{"Kinematics reconstruction":<28}{rmse_depth_mm(kin, GT):>10.1f}'
          f'{kc["aligned"]:>10.1f}{kc["shuffled"]:>10.1f}')
    print(f'{"MediaPipe native relative z":<28}{rmse_depth_mm(MP, GT):>10.1f}'
          f'{mc["aligned"]:>10.1f}{mc["shuffled"]:>10.1f}')
    print('-' * 58)
    print(f'{"Predict the mean (trivial)":<28}{"":>10}{kc["gt_mean"]:>10.1f}')
    print()

    # Verdict, stated only if the controls actually support one.
    floor = min(kc['gt_mean'], kc['shuffled'], mc['shuffled'])
    real  = {n: c['aligned'] for n, c in
             (('Kinematics', kc), ('MediaPipe native z', mc))
             if c['aligned'] < floor * 0.95}          # 5% clear of both nulls
    if not real:
        print('NO CANDIDATE BEATS THE CONTROLS.')
        print('Neither reconstruction is meaningfully better than shuffling the')
        print('predictions or than predicting a constant. Do not report a winner')
        print('from this table -- the honest result is that neither method')
        print('recovers wrist depth on this data.')
    else:
        best = min(real, key=real.get)
        print(f'Beats both controls by >5%: {", ".join(real)}')
        print(f'Lower is better -> {best} ({real[best]:.1f} mm)')

    print()
    print(f'paired samples: {len(GT)}   dropped for no depth: '
          f'{totals["dropped_no_depth"]}')
    print('Sensor quantises to ~10mm steps (Y11 is integral cm), so differences')
    print('below that are not meaningful.')

=== depth reconstruction vs measured depth (Z RMSE, mm) ===
Method                             raw   aligned  shuffled
----------------------------------------------------------
Kinematics reconstruction        826.5     367.8     384.2
MediaPipe native relative z     2282.6     372.9     384.1
----------------------------------------------------------
Predict the mean (trivial)                 384.2

NO CANDIDATE BEATS THE CONTROLS.
Neither reconstruction is meaningfully better than shuffling the
predictions or than predicting a constant. Do not report a winner
from this table -- the honest result is that neither method
recovers wrist depth on this data.

paired samples: 2124   dropped for no depth: 1149
Sensor quantises to ~10mm steps (Y11 is integral cm), so differences
below that are not meaningful.


## Summary

- [ ] Paired sample count is a reasonable fraction of detected frames. A large
      `dropped_no_depth` means the operator was outside the sensor's ~60 cm
      minimum range for much of the recording.
- [ ] **Check the controls before quoting any number.** The `shuffled` column
      is the one that matters: a candidate scoring near its own shuffle is not
      reconstructing anything, however good its RMSE looks.
- [ ] If no candidate beats the controls, report that as the finding. It is a
      legitimate negative result: it says a 6-DOF arm's forward kinematics,
      driven by a heuristic landmark-to-joint map with no extrinsic
      calibration, does not recover metric wrist depth. That is worth stating
      plainly rather than dressing up.
- [ ] Note the scale mismatch if you see it: the robot's reachable workspace
      is roughly 0.2 m, while an operator's wrist sweeps over 1 m. The
      kinematics reconstruction is confined to the former by the joint limits,
      so it cannot reproduce the latter's range no matter how good the
      landmarks are.